# Lesson 07 - Planning Design Pattern

Dis notebook dey show how **Planning Design Pattern** de work for AI agents wey dey use Microsoft Agent Framework.
You go sabi how to break complex travel requests into organized small-small tasks, assign am to specialist agents,
and execute di plan wey come out — all dis na with structured output wey Pydantic models dey power.


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Task Decomposition

Task decomposition na di core of di planning design pattern. Instead make you ask only one agent
to handle one complex request from start to finish, we go break di wahala into smaller, well-defined **subtasks**.
Every subtask dem go assign am to specialist agent (like flights, hotels, activities) wey get clear
priorities and order wey dem suppose follow.

Dis approach get plenti beta tins:
- **Clarity**: every subtask get one clear responsibility.
- **Parallelism**: subtasks wey no depend on each oda fit run at di same time.
- **Reliability**: if one subtask fail, e no go affect all di oda ones.
- **Budget tracking**: cost dey estimate for every subtask and dem dey add am up.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Di Planing Agent Wey Dey Get Structured Output

Di planing agent bi like **front desk coordinator**. If pesin give am high-level travel request, e
go produce structured `TravelPlan` — e go break di request into small small task dem, put priority,
and sabi di dependency so dat concierge or execution layer fit do di work.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## How to Dey Run Plan Wit Specialist Tools

After di front desk agent don produce one structured plan, na di **concierge agent** go run am.
Every specialist tool dey handle one kain subtask category (flights, hotels, activities). Di concierge
go follow di plan subtasks one by one for correct order and send each one go di
right tool.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Summary

For dis lesson, you don learn di **Planning Design Pattern** for AI agents:

1. **Task Decomposition** — One front desk planning agent go break one complex travel request into
   structured subtasks wey e go take Pydantic models, to assign each one to one specialist agent with priorities
   and dependencies.
2. **Structured Output** — By to pass `response_format` the agent go return one validated
   `TravelPlan` object wey no be free-form text, dis go make downstream processing dey reliable.
3. **Plan Execution** — One concierge agent go dey go through all the subtasks by to use specialist tools
   (`book_flight`, `reserve_hotel`, `book_activity`) to run di plan and give report for the results.

Dis pattern dey separate *wetin to do* (planning) from *how to do am* (execution), e dey make agents
more modular, easy to test, and easy to extend.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
